# Chapters 4 & 5 — Underfitting + Double Exponential Smoothing (Holt's Method)
## Case Study: E-commerce Fashion — Capturing the Growth Trend

**Reference:** Vandeput (2021), Chapters 4 & 5  
**Author:** Hanzila Bin Younus — EM CDE, University of Salzburg

---

### Industry Scenario

> **Company:** Fast-growing European e-commerce fashion retailer  
> **Problem:** The planning team has been using SES. MAE looks acceptable but  
> inventory is consistently running out during peak periods. Root cause: SES  
> only models the **level** — it completely misses the 15% month-over-month growth trend.  
> This is **underfitting**: the model is too simple for the demand pattern.

### Holt's Method — Two Equations
$$L_t = \alpha \cdot d_t + (1-\alpha)(L_{t-1} + T_{t-1})$$
$$T_t = \beta(L_t - L_{t-1}) + (1-\beta)T_{t-1}$$
$$f_{t+m} = L_t + m \cdot T_t$$

Where $L$ = level, $T$ = trend, $\alpha$ = level smoothing, $\beta$ = trend smoothing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fb',
                     'axes.spines.top': False, 'axes.spines.right': False})
C = {'demand': '#1a2332', 'ses': '#94a3b8', 'holt': '#0077b6',
     'level': '#6246ea', 'trend': '#e07b00'}

In [ ]:
# ── 1. E-COMMERCE TRENDING DEMAND ─────────────────────────────────────
np.random.seed(13)
n = 60
t = np.arange(n)

# Strong growth trend + small noise (fast-growing fashion category)
demand = 2000 + 35*t + np.random.normal(0, 80, n)
demand = np.clip(demand, 0, None)

TRAIN_END = 48; EXTRA = 12
print(f'Demand range: {demand.min():.0f} — {demand.max():.0f}')
print(f'Avg monthly growth: ~{(demand[12:]-demand[:-12]).mean()/demand[:-12].mean()*100:.1f}%')

In [ ]:
# ── 2. MODEL IMPLEMENTATIONS ─────────────────────────────────────────
def ses(d, alpha=0.3, extra=12):
    cols=len(d); f=np.full(cols+extra,np.nan); a=np.full(cols+extra,np.nan)
    a[0]=d[0]; f[1]=a[0]
    for i in range(1,cols):
        a[i]=alpha*d[i]+(1-alpha)*a[i-1]
        if i+1<cols+extra: f[i+1]=a[i]
    f[cols:]=a[cols-1]; return f

def holts_linear(d, alpha=0.3, beta=0.2, extra=12):
    """
    Holt's Linear Trend — Double Exponential Smoothing.
    Captures both level AND trend in the demand.
    Returns: forecast, level, trend arrays
    """
    cols=len(d)
    f=np.full(cols+extra,np.nan)
    L=np.full(cols+extra,np.nan)  # level
    T=np.full(cols+extra,np.nan)  # trend
    # Initialise
    L[0]=d[0]; T[0]=d[1]-d[0] if len(d)>1 else 0
    for i in range(1,cols):
        f[i]=L[i-1]+T[i-1]
        L[i]=alpha*d[i]+(1-alpha)*(L[i-1]+T[i-1])
        T[i]=beta*(L[i]-L[i-1])+(1-beta)*T[i-1]
    for m in range(1,extra+1):
        f[cols+m-1]=L[cols-1]+m*T[cols-1]
    return f, L, T

f_ses     = ses(demand, alpha=0.30, extra=EXTRA)
f_holt,L,T= holts_linear(demand, alpha=0.30, beta=0.20, extra=EXTRA)

def test_mae(d, f, te):
    a=d[te:]; fc=f[te:te+len(a)]
    m=~(np.isnan(a)|np.isnan(fc))
    return np.abs((fc-a)[m]).mean()

print(f'Test MAE — SES: {test_mae(demand,f_ses,TRAIN_END):.0f} units')
print(f'Test MAE — Holt: {test_mae(demand,f_holt,TRAIN_END):.0f} units')

In [ ]:
# ── 3. UNDERFITTING DIAGNOSIS ─────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 11),
                          gridspec_kw={'height_ratios':[3,1,1]})

ax = axes[0]
x  = np.arange(len(demand)+EXTRA)
ax.axvspan(0, TRAIN_END, alpha=0.05, color='#0077b6')
ax.axvspan(TRAIN_END, len(demand), alpha=0.05, color='#6246ea')
ax.axvline(TRAIN_END, color='#6246ea', lw=1.2, ls='--', alpha=0.5)

ax.plot(np.arange(len(demand)), demand, color=C['demand'], lw=2,
        label='Actual Demand', zorder=5)
ax.plot(x[:len(f_ses)],  f_ses,  color=C['ses'],  lw=1.8,
        ls='--', label=f"SES (underfitting — MAE={test_mae(demand,f_ses,TRAIN_END):.0f})")
ax.plot(x[:len(f_holt)], f_holt, color=C['holt'], lw=1.8,
        ls='--', label=f"Holt's Linear (MAE={test_mae(demand,f_holt,TRAIN_END):.0f})")

# Show SES lagging behind trend
ax.fill_between(np.arange(len(demand)), demand, f_ses[:len(demand)],
                where=demand > f_ses[:len(demand)],
                alpha=0.12, color='#c0392b', label='SES under-forecast (stockout risk)')

ax.set_title("Underfitting: SES Misses the Growth Trend — Holt's Captures It",
             fontsize=13, fontweight='bold')
ax.set_ylabel('Monthly Orders (units)'); ax.legend(fontsize=9)
ax.text(2, demand.max()*0.93, 'TRAIN', color='#0077b6', fontsize=9, fontweight='bold')
ax.text(TRAIN_END+0.5, demand.max()*0.93, 'TEST', color='#6246ea', fontsize=9, fontweight='bold')

# Level and Trend decomposition
ax2 = axes[1]
ax2.plot(L[:len(demand)], color=C['level'], lw=1.5, label='Level (L)')
ax2.plot(demand, color=C['demand'], lw=1.2, alpha=0.4, ls=':')
ax2.set_ylabel('Level'); ax2.legend(fontsize=8)
ax2.set_title("Holt's Decomposition — Level and Trend tracked separately", fontsize=9)

ax3 = axes[2]
ax3.plot(T[:len(demand)], color=C['trend'], lw=1.5, label='Trend (T)')
ax3.axhline(0, color='black', lw=0.8, ls='--')
ax3.set_ylabel('Trend'); ax3.set_xlabel('Month'); ax3.legend(fontsize=8)
ax3.set_title('Estimated trend per period (steady ~35 units/month)', fontsize=9)

plt.tight_layout()
plt.savefig('output_ch04_05_underfitting_holts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output_ch04_05_underfitting_holts.png')

In [ ]:
# ── 4. BETA SENSITIVITY ───────────────────────────────────────────────
betas = [0.05, 0.20, 0.50, 0.90]
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(demand)+EXTRA)

ax.plot(np.arange(len(demand)), demand, color=C['demand'],
        lw=2, label='Actual Demand', zorder=5)
ax.axvline(TRAIN_END, color='#6246ea', lw=1.2, ls='--', alpha=0.5)

beta_colors = ['#6246ea','#0077b6','#e07b00','#c0392b']
for beta, col in zip(betas, beta_colors):
    f_b,_,T_b = holts_linear(demand, alpha=0.30, beta=beta, extra=EXTRA)
    mae_b = test_mae(demand, f_b, TRAIN_END)
    ax.plot(x[:len(f_b)], f_b, color=col, lw=1.6, ls='--',
            label=f'β={beta} (MAE={mae_b:.0f})')

ax.set_title("Beta Sensitivity — How Fast Should the Trend Estimate Update?",
             fontsize=12, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Units'); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('output_ch05_beta_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output_ch05_beta_sensitivity.png')

In [ ]:
# ── 5. BUSINESS IMPACT QUANTIFICATION ────────────────────────────────
# How much does the underfitting cost in business terms?
test_demand   = demand[TRAIN_END:]
test_ses      = f_ses[TRAIN_END:TRAIN_END+len(test_demand)]
test_holt     = f_holt[TRAIN_END:TRAIN_END+len(test_demand)]

# Assumptions
unit_margin    = 12.0  # EUR profit per unit sold
holding_cost   = 2.5   # EUR cost per unit of excess stock per month

under_ses  = np.maximum(test_demand - test_ses, 0)   # lost sales
over_ses   = np.maximum(test_ses - test_demand, 0)   # overstock
under_holt = np.maximum(test_demand - test_holt, 0)
over_holt  = np.maximum(test_holt - test_demand, 0)

cost_ses  = under_ses.sum()*unit_margin + over_ses.sum()*holding_cost
cost_holt = under_holt.sum()*unit_margin + over_holt.sum()*holding_cost

print('=== BUSINESS IMPACT OF UNDERFITTING ===')
print(f'  Test period: {len(test_demand)} months')
print(f'  SES  — lost sales: {under_ses.sum():.0f} units | overstock: {over_ses.sum():.0f} units | cost: €{cost_ses:,.0f}')
print(f"  Holt — lost sales: {under_holt.sum():.0f} units | overstock: {over_holt.sum():.0f} units | cost: €{cost_holt:,.0f}")
print(f"  Improvement with Holt's: €{cost_ses-cost_holt:,.0f} saved ({(cost_ses-cost_holt)/cost_ses*100:.0f}% reduction)")

## Summary

| Model | Captures | Fails When | This Case |
|-------|---------|------------|----------|
| SES | Level only | Demand has trend | Underfits — systematic stockouts |
| Holt's | Level + Trend | Trend reverses suddenly | Correct model for growing demand |

**Underfitting diagnosis:** If residuals show a systematic positive or negative pattern  
(not random around zero), the model is too simple. Add complexity.

**Next:** `05_model_optimisation.ipynb` — automatic parameter search using grid search  
**After that:** `06_damped_trend.ipynb` — preventing Holt's from projecting infinite growth